# Zava agentic reinforcement fine-tuning

This notebook evaluates a tool-using `o4-mini` baseline, submits a fresh reinforcement fine-tuning job, deploys the resulting model, and compares both models on the same validation scenarios. The system prompt, tool definition, grader, training data, and validation data remain fixed so the model weights are the only experimental variable.

## Configuration

Set `AZURE_AI_PROJECT_ENDPOINT`, `ZAVA_TOOL_BASE_URL`, `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, and `AZURE_AOAI_ACCOUNT`. The tool service must be publicly reachable during training. `ZAVA_EVAL_LIMIT` can reduce evaluation cost for a smoke run; omit it to use all 57 validation scenarios.

In [ ]:
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentModel, DeploymentProperties, Sku

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RFT_DIR = ROOT / "rft"
sys.path.insert(0, str(RFT_DIR))

from calibrate import run_agent
from grader import get_grader_source, grade

project_endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"].rstrip("/")
tool_base_url = os.environ["ZAVA_TOOL_BASE_URL"].rstrip("/")
base_deployment = os.getenv("RFT_BASE_MODEL", "o4-mini")
deployment_name = os.getenv("RFT_DEPLOYMENT", "zava-rft")
eval_limit = int(os.getenv("ZAVA_EVAL_LIMIT", "57"))
eval_workers = int(os.getenv("ZAVA_EVAL_WORKERS", "3"))

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
client = project_client.get_openai_client()

train_path = RFT_DIR / "data" / "zava_v1_train.jsonl"
val_path = RFT_DIR / "data" / "zava_v1_val.jsonl"
validation_rows = [json.loads(line) for line in val_path.read_text(encoding="utf-8").splitlines() if line.strip()][:eval_limit]
print(f"Loaded {len(validation_rows)} validation scenarios")

In [ ]:
def evaluate(model):
    cache_dir = ROOT / "run"
    cache_dir.mkdir(exist_ok=True)
    safe_name = "".join(char if char.isalnum() else "-" for char in model)
    cache_path = cache_dir / f"{safe_name}-evaluation.json"
    scores = json.loads(cache_path.read_text()) if cache_path.exists() else []
    def evaluate_row(index_and_row):
        index, row = index_and_row
        for attempt in range(3):
            try:
                output, tool_calls = run_agent(client, row["messages"], model=model)
                score = grade({"output_text": output, "output_tools": tool_calls}, row)
                print(f"{model}: {index}/{len(validation_rows)} score={score:.3f}")
                return score
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 ** attempt)

    remaining = list(enumerate(validation_rows[len(scores):], start=len(scores) + 1))
    for offset in range(0, len(remaining), eval_workers):
        batch = remaining[offset:offset + eval_workers]
        with ThreadPoolExecutor(max_workers=eval_workers) as executor:
            scores.extend(executor.map(evaluate_row, batch))
        cache_path.write_text(json.dumps(scores), encoding="utf-8")
    threshold = 0.8
    return {
        "mean_score": sum(scores) / len(scores),
        "pass_rate": sum(score >= threshold for score in scores) / len(scores),
        "scores": scores,
    }

base_result = evaluate(base_deployment)
print(json.dumps(base_result, indent=2))

## Submit reinforcement fine-tuning

The default path uploads the bundled 343-row training set and 57-row validation set, then submits a new job. `ZAVA_RFT_JOB_ID` is available only for resuming a job created by an interrupted run of this notebook.

In [ ]:
def upload(path):
    with path.open("rb") as handle:
        uploaded = client.files.create(file=handle, purpose="fine-tune")
    deadline = time.time() + 1800
    while time.time() < deadline:
        try:
            status = client.files.retrieve(uploaded.id)
            if status.status == "processed":
                return uploaded.id
            if status.status in {"error", "failed"}:
                raise RuntimeError(f"File processing failed: {status}")
        except Exception as exc:
            print(f"File status check will retry: {type(exc).__name__}")
        time.sleep(10)
    raise TimeoutError(f"File {uploaded.id} was not processed")

resume_job_id = os.getenv("ZAVA_RFT_JOB_ID")
if resume_job_id:
    job = client.fine_tuning.jobs.retrieve(resume_job_id)
else:
    train_file_id = upload(train_path)
    val_file_id = upload(val_path)
    job = client.fine_tuning.jobs.create(
        model="o4-mini",
        training_file=train_file_id,
        validation_file=val_file_id,
        suffix="zava-agentic-rft",
        method={
            "type": "reinforcement",
            "reinforcement": {
                "grader": {
                    "type": "python",
                    "name": "zava_grader",
                    "source": get_grader_source(),
                    "pass_threshold": 0.8
                },
                "tools": [{
                    "name": "get_order",
                    "server_url": f"{tool_base_url}/tool/get_order",
                    "headers": {}
                }],
                "max_episode_steps": 5,
                "hyperparameters": {
                    "n_epochs": 3,
                    "learning_rate_multiplier": 1.0,
                    "compute_multiplier": 1.5,
                    "reasoning_effort": "medium",
                    "eval_interval": 5,
                    "eval_samples": 10
                }
            }
        },
        extra_body={"trainingType": "GlobalStandard"}
    )

print(f"Job: {job.id} ({job.status})")

In [ ]:
terminal_states = {"succeeded", "failed", "cancelled"}
while job.status not in terminal_states:
    time.sleep(30)
    for attempt in range(5):
        try:
            job = client.fine_tuning.jobs.retrieve(job.id)
            break
        except Exception:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)
    print(f"Training status: {job.status}")

if job.status != "succeeded":
    raise RuntimeError(f"RFT ended with status {job.status}: {job.error}")

fine_tuned_model = job.fine_tuned_model
print(f"Fine-tuned model: {fine_tuned_model}")

## Deploy and evaluate

The fine-tuned model is deployed to the same Foundry resource and evaluated with the same prompt, tools, scenarios, and grader used for the baseline.

In [ ]:
subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]
resource_group = os.environ["AZURE_RESOURCE_GROUP"]
account_name = os.environ["AZURE_AOAI_ACCOUNT"]
capacity = int(os.getenv("RFT_DEPLOYMENT_CAPACITY", "100"))

with CognitiveServicesManagementClient(credential, subscription_id) as management_client:
    deployment = Deployment(
        properties=DeploymentProperties(
            model=DeploymentModel(format="OpenAI", name=fine_tuned_model, version="1")
        ),
        sku=Sku(name="GlobalStandard", capacity=capacity),
    )
    management_client.deployments.begin_create_or_update(
        resource_group,
        account_name,
        deployment_name,
        deployment,
    ).result()

print(f"Deployment ready: {deployment_name}")

In [ ]:
fine_tuned_result = evaluate(deployment_name)

print("\nResult")
print(f"Base mean score:       {base_result['mean_score']:.3f}")
print(f"Base pass rate:        {base_result['pass_rate']:.0%}")
print(f"Fine-tuned mean score: {fine_tuned_result['mean_score']:.3f}")
print(f"Fine-tuned pass rate:  {fine_tuned_result['pass_rate']:.0%}")
print(f"Evaluation scenarios:  {len(validation_rows)}")

Delete the fine-tuned deployment when it is no longer needed. Training files and jobs remain in the Foundry project for reproducibility and can be removed through the project API or portal.